# When diffusion becomes numerically unstable

This notebook solves one-dimensional Fickian diffusion with the same conservative, cell-centred explicit scheme used on the MSE Learning Lab website. Change the Fourier number and rerun the experiment to see why

$$Fo = \frac{D\,\Delta t}{\Delta x^2} \leq \frac{1}{2}$$

is required for stability in one dimension. No installation is needed in Google Colab.

## 1. Choose the experiment

Start with **Fo = 0.45**. Then try **0.50**, **0.51**, and **0.55**. The code deliberately stops once the oscillation leaves the plotted range.

In [ ]:
# @title Experiment controls
Fo = 0.45  # @param {type:"slider", min:0.05, max:0.70, step:0.01}
nodes = 101  # @param [51, 101, 201] {type:"raw"}
target_tau = 0.08  # @param {type:"slider", min:0.005, max:0.20, step:0.005}
initial_profile = "step"  # @param ["step", "pulse", "linear"]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PLOT_MIN, PLOT_MAX = -2.0, 3.0
dx = 1.0 / nodes
x = -0.5 + (np.arange(nodes) + 0.5) * dx

if initial_profile == "pulse":
    concentration = np.zeros(nodes)
    centre = nodes // 2
    half_width = max(0, round(nodes * 0.01))
    concentration[centre - half_width:centre + half_width + 1] = 1.0
elif initial_profile == "linear":
    concentration = np.linspace(0.0, 1.0, nodes)
else:
    concentration = np.where(x < 0.0, 0.0, 1.0)

initial = concentration.copy()
initial_mass = concentration.sum()
delta_tau = Fo * dx**2
tau = 0.0
iteration = 0
diverged = False
amplitude_history = [np.max(np.abs(concentration - 0.5))]
tau_history = [0.0]

while tau < target_tau - 1e-15:
    step_tau = min(delta_tau, target_tau - tau)
    step_fo = step_tau / dx**2
    next_concentration = concentration.copy()

    # Zero boundary-face fluxes and the conservative FTCS balance.
    next_concentration[0] = concentration[0] + step_fo * (concentration[1] - concentration[0])
    next_concentration[1:-1] = concentration[1:-1] + step_fo * (
        concentration[2:] - 2.0 * concentration[1:-1] + concentration[:-2]
    )
    next_concentration[-1] = concentration[-1] + step_fo * (concentration[-2] - concentration[-1])

    concentration = next_concentration
    tau += step_tau
    iteration += 1
    amplitude_history.append(np.max(np.abs(concentration - 0.5)))
    tau_history.append(tau)

    if (
        not np.all(np.isfinite(concentration))
        or concentration.min() < PLOT_MIN
        or concentration.max() > PLOT_MAX
    ):
        diverged = True
        break

mass_percent = 100.0 * concentration.sum() / initial_mass
prediction = "stable" if Fo <= 0.5 else "unstable"
print(f"Fo = {Fo:.2f} -> predicted {prediction}")
print(f"Stopped at tau = {tau:.5f} after {iteration} increments")
print(f"Mass retained = {mass_percent:.12f}%")
print(f"Concentration range = [{concentration.min():.4f}, {concentration.max():.4f}]")
if diverged:
    print("Numerical divergence detected: the oscillation left the displayed range.")
elif Fo > 0.5:
    print("Fo exceeds 0.5; increase target_tau if the growing mode is not yet obvious.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].axhspan(0, 1, color="tab:green", alpha=0.07, label="physical range")
axes[0].plot(x, initial, "--", color="tab:orange", label="initial")
axes[0].plot(x, concentration, color=("tab:red" if Fo > 0.5 else "tab:blue"), label="numerical")
axes[0].set(xlabel="x / L", ylabel="Normalized concentration", title=f"Profile at τ = {tau:.5f}")
axes[0].set_ylim((PLOT_MIN, PLOT_MAX) if Fo > 0.5 else (-0.03, 1.03))
axes[0].legend()

axes[1].semilogy(tau_history, amplitude_history, color=("tab:red" if Fo > 0.5 else "tab:blue"))
axes[1].set(xlabel="Dimensionless time, τ", ylabel="max |C - 0.5|", title="Growth or decay of the strongest deviation")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 2. Questions to investigate

1. Compare Fo = 0.49, 0.50, and 0.51. How sharply does the behavior change?
2. Why can the reported mass remain close to 100% while negative and greater-than-one concentrations grow?
3. Change the number of cells while holding Fo fixed. Why does stability depend on Fo rather than on Δt alone?
4. Which feature of the unstable profile tells you immediately that it cannot represent ordinary diffusion?